In [1]:
# 0. Verisetini Hazırla
import pandas as pd
df = pd.read_csv("data/winequality_combined.csv")
df.head()

# Grover Dilemma: Type A'dan kaçınmak için 6497 satırlık verisetini bir 2 kuvvetine kırpıyoruz.
# Sebep: log2(6497) yaklaşık= 12.67, yani 13 kübit gerekir. Olası indis durumları: 2^13 = 8192 !! 8192 - 6497 = 1695 SAHTE İNDİS DEMEK !!
# Sahte indisler ölçümde veride olmayan satırlar döndürebileceği için satır sayısını tam bir 2 kuvvetine eşledik;
# böylece Type A Grover Dilemma'sından kurtulmuş olduk. (Bulut testleri için 2^3 = 8; yerelde 2^6 = 64 doğrulandı.)
df_ = df.copy().iloc[:8]
print("Ön işleme öncesi:")
print(df_)

alcohol_ideal_decimal = 2
density_ideal_decimal = 4
scale = {"alcohol": 10**alcohol_ideal_decimal, "density": 10**density_ideal_decimal}
print("Değişkenlerin en fazla alabildiği ondalık terim sayısı:")
for col in [col for col in df.columns if col not in ["alcohol", "density", "type"]]:
    max_decimal = df_[col].astype(str).str.split('.').str[1].fillna('').str.len().max()
    print("",col, max_decimal)
    df_[col] = (df[col].round(max_decimal) * (10 ** max_decimal)).round().astype(int)
    scale[col] = 10 ** max_decimal 
df_["alcohol"] = (df["alcohol"].round(alcohol_ideal_decimal) * scale["alcohol"]).round().astype(int)
df_["density"] = (df["density"].round(density_ideal_decimal) * scale["density"]).round().astype(int)
## 'type' ikili kategorik değişken (red/white) olduğu için 0 veya 1 değerleri atansa yeterli.
df_["type"] = (df_["type"]=="white").astype(int)
scale["type"] = 1
print("Veriler hazır. Veriseti artık QROM + sorgu oracle'ı boru hattı için hazır.")
print(df_)

Ön işleme öncesi:
   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   
5            7.4              0.66         0.00             1.8      0.075   
6            7.9              0.60         0.06             1.6      0.069   
7            7.3              0.65         0.00             1.2      0.065   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0                  54.0   0.9970  3.

In [2]:
print("Aradığımız değer:")
df_[(df_["fixed acidity"]==73) & (df_["quality"]>=7)]

Aradığımız değer:


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
7,73,65,0,12,65,150,210,9946,339,47,1000,7,0


In [4]:
# 1. Python + QDK entegrasyonu gerçek bir Azure Workspace'i ile uygulanmaya hazır hale getiriliyor.

# Varsayılan olarak 'rigetti.sim.qvm' devre simülatörü seçilmiştir. Dilerseniz farklı bir simülasyon-
# veya fiziksel Kuantum devreleri seçebilirsiniz. 
# Not: Azure ortamında çalışmak için Azure'a kayıt olunmalı, 'Quantum Workspace' sorgusu sonrası ilgili-
# adımları uygulayarak Kuantum Çalışma Alanı oluşturulmalı ve dilediğiniz devre API'ını planınıza dahil-
# edip işlemleri tamamlamalısınız. Sonra ilgili Çalışma Alanına gidip 'Resource ID' parametresini kopya-
# layıp '.env' dosyası içerisindeki 'RESOURCE_ID' alanına yapıştırın.

from qdk import qsharp
from azure.quantum import Workspace
from dotenv import load_dotenv
import os

load_dotenv()
qsharp.init(project_root=".", target_profile=qsharp.TargetProfile.Base)
resource_id = os.getenv("RESOURCE_ID")
# print("Azure Çalışma Ortamı Kaynak Kimliği:",resource_id + "\n")
workspace = Workspace(resource_id=resource_id)
targets = workspace.get_targets()
print("### Mevcut Çalıştırma Hedefleri ###")
for i, t in enumerate(targets, 1):
    print(f"{i}. Hedef:\n İsim: {t.name}\n Ort. Gecikme(s->saniye): {t.average_queue_time}\n Durum:{'MEVCUT' if t.current_availability == 'Available' else 'KULLANIM DIŞI'}")
target_str = "quantinuum.sim.h2-1e" #"rigetti.sim.qvm" "quantinuum.sim.h2-1sc"

### Mevcut Çalıştırma Hedefleri ###
1. Hedef:
 İsim: rigetti.sim.qvm
 Ort. Gecikme(s->saniye): 5
 Durum:MEVCUT
2. Hedef:
 İsim: quantinuum.sim.h2-1sc
 Ort. Gecikme(s->saniye): 0
 Durum:MEVCUT
3. Hedef:
 İsim: quantinuum.sim.h2-1e
 Ort. Gecikme(s->saniye): 6721
 Durum:MEVCUT


In [5]:
# 2. Grover Algoritması içerisine gönderilecek olan değişken ve aritmetik işlemler Q#'a uygun hale getirmek için ara işlemler uyguluyoruz.
# Hem yerelde hem de Azure Bulut için sütunları küçültebiliriz. Böylece her satır ve sütun için gereken kübitler azalır.

COMP_OPS = {"==": 0, ">": 1, "<": 2, ">=": 3, "<=": 4, "!=":5}
LOGIC_OPS = {"AND":0, "OR":1}
# column_order = ["fixed acidity", "volatile acidity", "citric acid",
#                 "residual sugar", "chlorides", "free sulfur dioxide",
#                 "total sulfur dioxide", "pH", "sulphates",
#                 "alcohol", "density", "quality", "type"]
column_order = ["fixed acidity", "quality"]
print("Karşılaştırma operatörlerine denk gelen indisler:")
for op, i in COMP_OPS.items():
    print(f" {op}: {i}.indis")
print()
print("Mantıksal operatörlere denk gelen indisler:")
for op, i in LOGIC_OPS.items():
    print(f" {op}: {i}.indis")


Karşılaştırma operatörlerine denk gelen indisler:
 ==: 0.indis
 >: 1.indis
 <: 2.indis
 >=: 3.indis
 <=: 4.indis
 !=: 5.indis

Mantıksal operatörlere denk gelen indisler:
 AND: 0.indis
 OR: 1.indis


In [6]:
# 3. Her bir sütun için bit genişliği bilgilerini oluşturup saklamalıyız. Böylece Q#'ta belirli sorgu için yapılacak olan kübit işlemleri doğru şekilde
# uygulanmış olur

weight = {}
print(f"Sütunların bit genişlikleri:")
for col in column_order:
    max_val = int(df_[col].max())
    
    bit_length = max_val.bit_length()
    print(f" {col}: {bit_length}")
    weight[col] = bit_length

def row_to_bits(satir):
    all_bits = []

    for col in column_order:
        value = int(satir[col])
        
        bit_count = weight[col]

        # DİKKAT (endianness): Q# tarafındaki ApplyControlledOnInt, dilimin 0. kübitini
        # en düşük anlamlı bit (LSB) olarak okur (little-endian). format() ise MSB-first
        # üretir; bu yüzden diziyi ters çevirerek ekliyoruz. Ters çevirmezsek palindrom
        # olmayan değerler (örn. 112 = 1110000) devrede yanlış sayıya dönüşür.
        binary_str = format(value, f"0{bit_count}b")
        for char in reversed(binary_str):
            if char == '1':
                all_bits.append(True)
            else:
                all_bits.append(False)

    return all_bits

dataset = df_[column_order].apply(row_to_bits, axis=1).tolist()

print("Veriseti Q#'ın anlayacağı biçimde oluşturuldu.\n")

# Her satırın devreye gidecek bit kombinasyonunu ayrıntılı gösterelim.
# Satır düzeni: [fixed acidity'nin 7 biti | quality'nin 3 biti] şeklinde art arda eklenmiştir.
# Her sütun dilimi little-endian tutulur: dilimin ilk elemanı LSB'dir.
for i, row_bits in enumerate(dataset):
    print(f"{i}. indisin bit kombinasyonu:")
    print(f"  Tamamı ({len(row_bits)} bit): {row_bits}")

    # Sütun dilimlerini gezmek için bir konum imleci tutuyoruz.
    # (offset sözlüğü bir sonraki hücrede tanımlandığı için burada imleçle ilerliyoruz.)
    position = 0
    for col in column_order:
        bit_count = weight[col]
        col_bits = row_bits[position : position + bit_count]

        # Little-endian dilimi tekrar tam sayıya çevirip doğrulama amaçlı gösteriyoruz.
        col_value = sum((1 << k) for k, bit in enumerate(col_bits) if bit)

        print(f"  {col} (konum {position}-{position + bit_count - 1}, {bit_count} bit, LSB önce): {col_bits} -> {col_value}")
        position += bit_count
    print()

# Yapısal bilgiler
print("Toplam satır sayısı:", len(dataset))

Sütunların bit genişlikleri:
 fixed acidity: 7
 quality: 3
Veriseti Q#'ın anlayacağı biçimde oluşturuldu.

0. indisin bit kombinasyonu:
  Tamamı (10 bit): [False, True, False, True, False, False, True, True, False, True]
  fixed acidity (konum 0-6, 7 bit, LSB önce): [False, True, False, True, False, False, True] -> 74
  quality (konum 7-9, 3 bit, LSB önce): [True, False, True] -> 5

1. indisin bit kombinasyonu:
  Tamamı (10 bit): [False, True, True, True, False, False, True, True, False, True]
  fixed acidity (konum 0-6, 7 bit, LSB önce): [False, True, True, True, False, False, True] -> 78
  quality (konum 7-9, 3 bit, LSB önce): [True, False, True] -> 5

2. indisin bit kombinasyonu:
  Tamamı (10 bit): [False, True, True, True, False, False, True, True, False, True]
  fixed acidity (konum 0-6, 7 bit, LSB önce): [False, True, True, True, False, False, True] -> 78
  quality (konum 7-9, 3 bit, LSB önce): [True, False, True] -> 5

3. indisin bit kombinasyonu:
  Tamamı (10 bit): [False, Fals

In [7]:
# 4. Q#'ın sütun değişkenlerini ayırt etmesi için bit-offset oluşturuyoruz.
offset = {}
loc = 0
print("Her bir sütunun bit-offset değeri:")
for col in column_order:
    offset[col]=loc
    print(f" {col}: {loc}")
    loc+=weight[col]

Her bir sütunun bit-offset değeri:
 fixed acidity: 0
 quality: 7


In [8]:
# 5. Q# içerisinde kullanmak için uyumlu bir betik sorgusu formatı oluşturuyoruz.
def q(col, op, deger):
    v = int(round(deger * scale[col]))
    assert v.bit_length() <= weight[col], f"{col}: {v} değeri {weight[col]} bite sığmıyor"
    return (offset[col], weight[col], COMP_OPS[op], v)

# Yapı: (mantıksal operatör, [karşılaştırma listesi])
# Q# tarafındaki karşılığı: (Int, (Int, Int, Int, Int)[])
# NOT: Dış yapı tuple olmak zorunda; Python tuple -> Q# tuple, Python liste -> Q# dizisi olarak eşlenir.
queries = (LOGIC_OPS["AND"], [
    q("fixed acidity", "==", 7.3),
    q("quality", ">=", 7)
])

# M: kaç eşleşme beklediğimize dair TAHMİN. Grover'ın tur sayısını belirler: pi/4 * sqrt(N/M).
# Cevabı pandas ile sayıp göndermiyoruz (o zaman aramanın anlamı kalmazdı); bu bir kullanıcı tahmini.
# Yanlış tahmin sonucu bozmaz ama olasılığı düşürür (aşırı/eksik dönme, souffle problemi).
expected_matches = 1

print("Fixed Acidity sütunu 7.3 olan VE quality sütunu 7 veya daha büyük değerler için Q# uyumlu sorgu betiği:")
print(queries)
print("Beklenen eşleşme tahmini (M):", expected_matches)

Fixed Acidity sütunu 7.3 olan VE quality sütunu 7 veya daha büyük değerler için Q# uyumlu sorgu betiği:
(0, [(0, 7, 0, 73), (7, 3, 3, 7)])
Beklenen eşleşme tahmini (M): 1


In [9]:
# 6.1 Q# devresine işlemler gönderilir ve hesaplama başlatılır. Aygıt: quantinuum.sim.h2-1e
# NOT: Sonuçlar VASAT çünkü bu aygıt gürültülü şekilde simüle ediyor. Temiz sonuç için 7.1'deki yerel doğrulama hücresine bakın.
try:
    print(f"'{target_str}' seçiliyor...")
    target = workspace.get_targets(target_str)
except Exception as e:
    print(f"Bilinmeyen bir hata oluştu:\n{e}")
print("Seçim tamamlandı. Derleme işlemine geçildi.\n")
app_class = "Main.GroverSearchAlgorithm"
op = qsharp.eval(app_class)
print(f"'{app_class}' Sınıfı derleniyor...")
program = qsharp.compile(op, queries, dataset, expected_matches)
print(f"Derleme bitti. İş akışı {target_str}'e gönderildi.\n")
job = target.submit(program, "MicrosoftFY26GroverJobQuantinuumNoisy", shots=10)
print("İş akışı gönderildi. Sonuçların yazdırılması bekleniyor... (5dk üst limit, olmazsa limitsiz fallback.)")
# print("İş akışı kimliği:",job.details.id)
try:
    results = job.get_results()
except TimeoutError:
    print("Timeout yedik. Limitsiz fallback uygulanıyor...")
    job.refresh()
    results = job.get_results(timeout_secs=None)
except RuntimeError as e:
    print("İş başarısız:", e)

print(results)

'quantinuum.sim.h2-1e' seçiliyor...
Seçim tamamlandı. Derleme işlemine geçildi.

'Main.GroverSearchAlgorithm' Sınıfı derleniyor...
Derleme bitti. İş akışı quantinuum.sim.h2-1e'e gönderildi.

İş akışı gönderildi. Sonuçların yazdırılması bekleniyor... (5dk üst limit, olmazsa limitsiz fallback.)
....................Timeout yedik. Limitsiz fallback uygulanıyor...
...............................................................................................{'[1, 0, 1]': 0.1, '[0, 0, 1]': 0.1, '[0, 1, 1]': 0.2, '[1, 1, 1]': 0.1, '[1, 1, 0]': 0.1, '[1, 0, 0]': 0.2, '[0, 0, 0]': 0.1, '[0, 1, 0]': 0.1}


In [ ]:
# 6.2 aynı test ancak farklı aygıt: rigetti.sim.qvm
# !BUG!
# DATE: 27/08/2026
target_str = "rigetti.sim.qvm"
try:
    print(f"'{target_str}' seçiliyor...")
    target = workspace.get_targets(target_str)
except Exception as e:
    print(f"Bilinmeyen bir hata oluştu:\n{e}")
print("Seçim tamamlandı. Derleme işlemine geçildi.\n")
print(f"'{app_class}' Sınıfı derleniyor...")
program = qsharp.compile(op, queries, dataset, expected_matches)
print(f"Derleme bitti. İş akışı {target_str}'e gönderildi.\n")
job = target.submit(program, "MicrosoftFY26GroverJobRigetti", shots=10)
print("İş akışı gönderildi. Sonuçların yazdırılması bekleniyor... (5dk üst limit, olmazsa limitsiz fallback.)")
print("İş akışı kimliği:",job.details.id)
try:
    results = job.get_results()
except TimeoutError:
    print("Timeout yedik. Limitsiz fallback uygulanıyor...")
    job.refresh()
    results = job.get_results(timeout_secs=None)
except RuntimeError as e:
    print("İş başarısız:", e)

print(results)

In [10]:
# 7. Sonuçları gözlemle.
import json
for bits, p in results.items():
    idx = sum(b << i for i, b in enumerate(json.loads(bits)))
    print(f"indeks {idx}: {p}")
    print("-"*5)
    print(df_.iloc[idx])
    print()


indeks 5: 0.1
-----
fixed acidity             74
volatile acidity          66
citric acid                0
residual sugar            18
chlorides                 75
free sulfur dioxide      130
total sulfur dioxide     400
density                 9978
pH                       351
sulphates                 56
alcohol                  940
quality                    5
type                       0
Name: 5, dtype: int64

indeks 4: 0.1
-----
fixed acidity             74
volatile acidity          70
citric acid                0
residual sugar            19
chlorides                 76
free sulfur dioxide      110
total sulfur dioxide     340
density                 9978
pH                       351
sulphates                 56
alcohol                  940
quality                    5
type                       0
Name: 4, dtype: int64

indeks 6: 0.2
-----
fixed acidity             79
volatile acidity          60
citric acid                6
residual sugar            16
chlorides               

In [11]:
# 7.1 BONUS: yerel doğrulama — Azure'a hiç gitmeden
from collections import Counter

def decode(shot):  # little-endian: ilk eleman LSB
    return sum((1 if str(b) == "One" else 0) << i for i, b in enumerate(shot))

app_class = "Main.GroverSearchAlgorithm"
op = qsharp.eval(app_class)
sonuclar = qsharp.run(op, 100, queries, dataset, expected_matches)
hist = Counter(decode(s) for s in sonuclar)

mask = (df_["fixed acidity"] == 73) & (df_["quality"] >= 7)
print(df_[mask].index.tolist())   # beklenen: [7]

print(hist.most_common(5))

[7]
[(7, 97), (0, 2), (1, 1)]


In [ ]:
## HATA AYIKLAMA ##

# qir = str(program)
# with open("program.ll", "w") as f: f.write(qir)

# import re
# from collections import Counter
# print(Counter(re.findall(r"__quantum__qis__(\w+?)__", qir)))
# print(re.findall(r'required_num_(qubits|results)"="(\d+)', qir))

# mini = qsharp.compile("{ use qs = Qubit[3]; CCNOT(qs[0], qs[1], qs[2]); MResetEachZ(qs) }")
# job2 = target.submit(mini, "ccx-testi", shots=10)
# job2.get_results()

In [ ]:
## AYRIK İŞ TAKİBİ ##.
# 'job.details.status' çıktıları: Waiting / Executing / Succeeded / Failed
# 'job_id': job.details.id
job = workspace.get_job("job_id")
print(job.details.status)          
print(job.get_results())